# Customer Segmentation & Churn Analysis

## Step 1: Data Loading

In this notebook we will:

- Import libraries
- Load the dataset
- Understand the dataset structure
- Inspect rows and columns

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../data/Online Retail.xlsx")

## Dataset Overview

In this section we will understand the structure of the dataset before cleaning it.

We will inspect:

- Shape
- Columns
- Data types
- Missing values
- Summary statistics

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.shape

(541909, 8)

In [5]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


### Observation

- Dataset contains **541,909 rows** and **8 columns**.
- The `Description` and `CustomerID` columns contain missing values.
- `InvoiceDate` is stored as **datetime**.
- `Quantity` is an integer, while `UnitPrice` and `CustomerID` are numeric.
- The dataset contains customer transaction records from multiple countries.

In [7]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


### Observation

- The average quantity purchased per transaction is **9.55**, while the median is **3**, which suggests that most orders contain only a few items.
- The average unit price is **4.61**, but the maximum unit price reaches **38,970**, indicating the presence of extreme values.
- I noticed negative values in both **Quantity** (minimum **-80,995**) and **UnitPrice** (minimum **-11,062.06**), which could represent cancelled orders, returns, or data entry errors.
- The maximum quantity of **80,995** is much higher than the average, suggesting there are significant outliers in the dataset.
- The data should be cleaned before further analysis to ensure more reliable insights.

In [8]:
df.describe(include="all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909.0,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900.0,4070,4223,NaN,NaN,NaN,NaN,38
top,573585.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1114.0,2313,2369,NaN,NaN,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:34:00,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-19 17:17:00,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 11:27:00,4.130000,16791.000000,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,NaN


## Observation

- The dataset contains **541,909 transactions** across **25,900 unique invoices**, indicating that each invoice can contain multiple products.
- There are **4,070 unique products**, providing a diverse product catalog for customer purchasing analysis.
- The most frequently purchased product is **WHITE HANGING HEART T-LIGHT HOLDER**, appearing **2,369 times**, making it one of the highest-demand products.
- Transactions come from **38 different countries**, although the business is heavily concentrated in the **United Kingdom**, which accounts for **495,478 transactions (over 91% of all records)**.
- Since the UK dominates the dataset, any overall business insights may be strongly influenced by customer behavior in this market.

In [9]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

### Observation

Missing values exist in:

- **Description**: 1,454 missing values.
- **CustomerID**: 135,080 missing values.

Since customer-based analysis requires a valid customer ID, rows with missing `CustomerID` will be handled during the data cleaning stage. Missing `Description` values will also be reviewed before further analysis.

In [10]:
df.duplicated().sum()

np.int64(5268)

### Observation

- The dataset contains **5,268 duplicate records**.

In [11]:
df.nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

### Observation

- The dataset contains **25,900 unique invoices**, **4,372 unique customers**, and **4,070 unique products**.
- Transactions were made across **38 countries**, indicating a global customer base.
- Many customers placed multiple orders, making the dataset suitable for customer behavior analysis.

In [12]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

### Observation

- The dataset contains **4 categorical columns** (`InvoiceNo`, `StockCode`, `Description`, `Country`).
- It has **3 numerical columns** (`Quantity`, `UnitPrice`, `CustomerID`) and **1 datetime column** (`InvoiceDate`).
- The `InvoiceDate` column can be used for time-based and customer behavior analysis.

In [13]:
print("Start Date :", df['InvoiceDate'].min())
print("End Date   :", df['InvoiceDate'].max())

Start Date : 2010-12-01 08:26:00
End Date   : 2011-12-09 12:50:00


### Observation

- The dataset covers transactions from **1 December 2010** to **9 December 2011**.
- It provides approximately **one year of purchase history**, which is sufficient to analyze customer purchasing patterns and perform RFM analysis.

In [14]:
print("Total Transactions :", df['InvoiceNo'].nunique())
print("Total Customers    :", df['CustomerID'].nunique())
print("Total Products     :", df['StockCode'].nunique())
print("Countries          :", df['Country'].nunique())

Total Transactions : 25900
Total Customers    : 4372
Total Products     : 4070
Countries          : 38
